# NYC Taxi & Weather Data Engineering Pipeline
### Medallion Architecture Implementation (Bronze -> Silver -> Gold)
**Tools:** PySpark, SQL, Delta Lake

## 1. Bronze Layer: Data Ingestion
*In this phase, we ingest raw data from NYC TLC (Parquet) and Open-Meteo API (CSV) into our landing zone.*

In [0]:
# Read the table created in the Catalog
taxis_df = spark.read.parquet("/Volumes/workspace/default/yello_taxi_nyc/Yellow Taxi Data/")

# Display the first 5 rows
display(taxis_df.limit(5))

In [0]:
# Read the weather file (CSV)
weather_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .option("comment", "#")
    .csv("/Volumes/workspace/default/yello_taxi_nyc/open-meteo-40.81N74.02W44m.csv")
)

# Display the first 5 rows of the weather data
display(weather_df.limit(5))

In [0]:
# Taxi data structure (Schema)
taxis_df.printSchema()

# Count the total number of records (Rows)
print(f"Total rows in taxi data: {taxis_df.count():,}")

In [0]:
# Weather data structure (Schema)
weather_df.printSchema()

# Count the total number of records (Rows)
print(f"Συνολικές γραμμές στα δεδομένα καιρού: {weather_df.count():,}")

## 2. Silver Layer: Data Cleaning & Transformation
*In this phase, we perform data quality checks, filter out anomalies, and enforce schema consistency.*
*   Removed trips with zero distance or negative amounts.
*   Casted data types to correct formats.
*   Normalized date formats for relational joining.


In [0]:
from pyspark.sql.functions import col, to_date

# Data Cleaning
taxis_silver_df = taxis_df.filter(
    (col("tpep_pickup_datetime") >= "2026-01-01 00:00:00") & 
    (col("tpep_pickup_datetime") <= "2026-03-31 23:59:59") & 
    (col("trip_distance") > 0) & 
    (col("total_amount") > 0) &
    (col("passenger_count") > 0)
)

# Create 'pickup_date' column (YYYY-MM-DD) to join with weather data
taxis_silver_df = taxis_silver_df.withColumn("pickup_date", to_date(col("tpep_pickup_datetime")))

# Checking how many "dirty" rows were dropped
original_count = 11_077_206
cleaned_count = taxis_silver_df.count()
deleted_rows = original_count - cleaned_count

print(f"Clean rows: {cleaned_count:,}")
print(f"Rows removed as invalid: {deleted_rows:,}")

In [0]:
from pyspark.sql.functions import col, to_date

# Keep only the rows that contain actual date records
weather_silver_df = weather_df.filter(
    (col("latitude") != "time") & 
    (col("latitude") != "40.808434")
)

# Select columns, rename them, and cast to appropriate data types
weather_silver_df = weather_silver_df.select(
    to_date(col("latitude")).alias("weather_date"),
    col("longitude").cast("double").alias("avg_temp"),
    col("elevation").cast("double").alias("precipitation")
)

# Display the cleaned weather dataframe
display(weather_silver_df)

## 3. Gold Layer: Data Enrichment & Joining
*Final stage where we join the two data sources to create a unified 'Gold' table for business intelligence.*

In [0]:
# Perform the JOIN based on the date
final_gold_df = taxis_silver_df.join(
    weather_silver_df, 
    taxis_silver_df["pickup_date"] == weather_silver_df["weather_date"], 
    "inner"
)

# Register the DataFrame as a temporary SQL view in Spark's memory
final_gold_df.createOrReplaceTempView("gold_nyc_taxi_weather")

display(final_gold_df.limit(10))

## 4. Business Intelligence & Analytics
*Querying the Gold layer to answer key business questions.*

### 🔍 Hypothesis 1: Weather Impact on Passenger Tipping Behavioral Patterns

> **Core Objective:** Investigate whether inclement weather conditions (rain/snow) increase passenger gratitude due to the difficulty of securing a ride, thereby driving higher average tip amounts.

In [0]:
%sql
SELECT 
    CASE 
        WHEN precipitation = 0 THEN 'No Rain (Dry Weather)'
        WHEN precipitation > 0 AND precipitation <= 2 THEN 'Drizzle / Light Rain'
        ELSE 'Storm / Heavy Rain'
    END AS weather_condition,
    COUNT(*) AS total_rides,
    ROUND(AVG(trip_distance), 2) AS avg_distance,
    ROUND(AVG(tip_amount), 2) AS avg_tip_amount,
    ROUND(AVG(total_amount), 2) AS avg_total_paid
FROM gold_nyc_taxi_weather
GROUP BY 1
ORDER BY avg_tip_amount DESC;

Databricks visualization. Run in Databricks to view.

### 💰 Tipping Analytics: Weather Impact on Passenger Generosity

An analysis of how precipitation levels influence tipping behavior across 5.8M+ rides reveals an unexpected trend:

* **Dry Weather:** 3.75M rides | Avg Tip: **$3.64**
* **Light Rain:** 216K rides | Avg Tip: **$3.62**
* **Heavy Rain:** 1.88M rides | Avg Tip: **$3.57**

**Key Finding:** Contrary to the initial hypothesis that passengers would tip more out of gratitude during bad weather, average tips actually **decrease** as rain intensity increases. This slight decline (~2% lower tips during heavy rain) is likely driven by higher total fare amounts due to traffic delays or increased passenger frustration caused by weather-related trip disruptions.


### 🔍 Hypothesis 2: Weather Impact on Urban Traffic Velocity 

> **Core Objective:** Determine if severe precipitation paralyzes New York City's infrastructure, leading to extended trip durations and a noticeable drop in average taxi speeds.

In [0]:
%sql
SELECT 
    CASE 
        WHEN precipitation = 0 THEN 'Dry Weather'
        ELSE 'Rain/Snow'
    END AS weather_type,
    COUNT(*) AS total_rides,
    ROUND(AVG((UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime)) / 60), 2) AS avg_duration_minutes,
    ROUND(AVG(trip_distance / ((UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime)) / 3600)), 2) AS avg_speed_mph
FROM gold_nyc_taxi_weather
WHERE (UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime)) BETWEEN 60 AND 18000
GROUP BY 1;

### 📊 Traffic Analytics: Weather Impact on Ride Performance

An analysis of the final dataset comparing dry vs. rainy/snowy weather conditions reveals a counter-intuitive pattern in NYC traffic dynamics:

* **Dry Weather:** 3.67M rides | Avg Duration: **17.76 mins** | Avg Speed: **10.41 mph**
* **Rain/Snow:** 3.97M rides | Avg Duration: **16.95 mins** | Avg Speed: **11.19 mph**

**Key Finding (The Traffic Paradox):** Taxis operate **~7.5% faster** with shorter trip durations during inclement weather. This paradox suggests that severe weather (such as winter snow or heavy rain) reduces overall amateur traffic volume on the streets, allowing experienced, professional taxi drivers to navigate NYC more efficiently.

# 5. Data Persistence & Storage
*Saving the Gold Layer to Delta Lake*

 * To ensure data durability and high-availability for downstream BI tools (e.g., Power BI, Tableau), the temporary memory view is persisted as a permanent ACID-compliant **Delta Table**.
 * Applied the overwrite mode to ensure the pipeline can refresh the data idempotently.

In [0]:
# Saving permanently the final table to the Catalog
(
    final_gold_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.default.gold_taxi_weather_analytics")
)

## 